# Cross-Camera Split

Construct a deterministic SeaClear-only split by camera/source. This assesses camera transfer within Bistrina, not location-independent detection.


## 1. Data paths

Locate the verified source data and the output directory.


In [ ]:
# Resolve source directories and verify that the input datasets are available.
from google.colab import drive
from pathlib import Path
from collections import Counter
import pandas as pd
import yaml

drive.mount('/content/drive')

ORIGINAL = Path('/content/drive/MyDrive/MarineGuard_Original_SeaClear/Seaclear Marine Debris Dataset')
CURRENT = Path('/content/drive/MyDrive/MarineGuard_YOLO/MarineGuard_YOLO_Fixed')
OUT = Path('/content/drive/MyDrive/MarineGuard_Validation_Improvement/MarineGuard_v2_verified_seaclear_manifest')
HOLDOUT_CAMERA_GROUP = 'Bistrina/Bluerobotics HD'
IMAGE_EXT = {'.jpg', '.jpeg', '.png', '.webp'}

assert (ORIGINAL / 'dataset.json').is_file(), 'Missing original SeaClear dataset.json'
assert (CURRENT / 'data.yaml').is_file(), 'Missing current dataset data.yaml'
for split in ('train', 'val'):
    assert (CURRENT / 'images' / split).is_dir(), f'Missing images/{split}'
    assert (CURRENT / 'labels' / split).is_dir(), f'Missing labels/{split}'
OUT.mkdir(parents=True, exist_ok=True)
print('Source dataset:', CURRENT)
print('Original SeaClear:', ORIGINAL)
print('Output:', OUT)

Mounted at /content/drive
Source dataset: /content/drive/MyDrive/MarineGuard_YOLO/MarineGuard_YOLO_Fixed
Original SeaClear: /content/drive/MyDrive/MarineGuard_Original_SeaClear/Seaclear Marine Debris Dataset
Output: /content/drive/MyDrive/MarineGuard_Validation_Improvement/MarineGuard_v2_verified_seaclear_manifest


## 2. Camera-group assignment

Trace image origins, check labels, and hold out complete camera groups.


In [ ]:
# Map image provenance, validate annotations, and report camera-disjoint splits.
# Create a one-to-one original filename -> site/camera map.
# Abort instead of guessing if an original filename occurs in multiple folders.
source_index = {}
for p in ORIGINAL.rglob('*'):
    if p.is_file() and p.suffix.lower() in IMAGE_EXT:
        key = p.name.lower()
        if key in source_index:
            raise RuntimeError(f'Ambiguous original filename: {p.name}')
        source_index[key] = str(p.relative_to(ORIGINAL).parent)

assert len(source_index) == 8610, f'Unexpected SeaClear image count: {len(source_index)}'
print('Original SeaClear images indexed:', len(source_index))

def read_label_counts(label_file):
    assert label_file.is_file(), f'Missing YOLO label file: {label_file}'
    counts = Counter()
    for i, line in enumerate(label_file.read_text(encoding='utf-8').splitlines(), start=1):
        if not line.strip():
            continue
        parts = line.split()
        assert len(parts) == 5, f'Malformed line {i} in {label_file}'
        cls = int(parts[0])
        assert cls in (0, 1), f'Unexpected class {cls} in {label_file}'
        xc, yc, w, h = [float(x) for x in parts[1:]]
        assert w > 0 and h > 0 and min(xc-w/2, yc-h/2) >= -1e-4 and max(xc+w/2, yc+h/2) <= 1+1e-4, f'Invalid box in {label_file}'
        counts[cls] += 1
    return counts

rows=[]
for old_split in ('train', 'val'):
    for img in sorted((CURRENT / 'images' / old_split).iterdir()):
        if not (img.is_file() and img.suffix.lower() in IMAGE_EXT):
            continue
        label = CURRENT / 'labels' / old_split / (img.stem + '.txt')
        counts = read_label_counts(label)
        original_source = source_index.get(img.name.lower())
        if original_source is not None:
            provenance = 'SeaClear_verified'
            # Assign splits after we have identified *all* source groups used by the old validation.
            new_split = 'PENDING'
        elif img.stem.lower().startswith('cleansea'):
            provenance = 'CleanSea_cutout_excluded'
            new_split = 'excluded'
        elif img.stem.lower().startswith('bg_'):
            provenance = 'Extra_background_unverified_excluded'
            new_split = 'excluded'
        else:
            provenance = 'UNKNOWN_EXCLUDED'
            new_split = 'excluded'
        rows.append(dict(old_split=old_split, proposed_split=new_split, provenance=provenance,
                         source_group=original_source or 'UNKNOWN', filename=img.name,
                         nets=counts[0], ropes=counts[1], image_path=str(img), label_path=str(label)))

inventory = pd.DataFrame(rows)
# If even one old Validation image belongs to a camera group, hold out every
# SeaClear image from that group, including images previously in Training.
old_val_groups = set(inventory.loc[(inventory.old_split == 'val') & (inventory.provenance == 'SeaClear_verified'), 'source_group'])
validation_groups = old_val_groups | {HOLDOUT_CAMERA_GROUP}
valid_source = inventory.provenance == 'SeaClear_verified'
inventory.loc[valid_source, 'proposed_split'] = inventory.loc[valid_source, 'source_group'].apply(
    lambda group: 'val' if group in validation_groups else 'train'
)
print('===== VALIDATION SOURCE GROUPS (ALL HELD OUT) =====')
for group in sorted(validation_groups):
    print(' -', group)
print('\n===== SOURCE CLASSIFICATION =====')
display(pd.crosstab(inventory['old_split'], inventory['provenance']))
print('\n===== UNKNOWN IMAGES =====')
unknown = inventory[inventory['provenance'] == 'UNKNOWN_EXCLUDED']
display(unknown[['old_split', 'filename']].head(20))
assert unknown.empty, 'Unknown input files found; investigate provenance before building a split.'
assert (inventory['provenance'] == 'SeaClear_verified').sum() == 2311, 'Unexpected SeaClear image total; examine audit results.'
assert set(inventory[inventory.old_split=='val'].provenance) == {'SeaClear_verified'}, 'Original validation contains unverified source images.'
assert (inventory.proposed_split == 'val').any() and (inventory.proposed_split == 'train').any()

# Ensure *no* original source-folder group is represented in both new splits.
kept = inventory[inventory.proposed_split.isin(('train','val'))].copy()
train_groups = set(kept[kept.proposed_split=='train'].source_group)
val_groups = set(kept[kept.proposed_split=='val'].source_group)
overlap = sorted(train_groups & val_groups)
print('\n===== SOURCE-CAMERA OVERLAP BETWEEN NEW SPLITS =====')
print(overlap if overlap else 'None — all SeaClear camera/source folders are disjoint')
assert not overlap, 'Source group overlaps Training and Validation; stop and revise split.'
assert HOLDOUT_CAMERA_GROUP in val_groups
assert HOLDOUT_CAMERA_GROUP not in train_groups

for split in ('train','val'):
    selected=kept[kept.proposed_split==split]
    assert selected.image_path.is_unique, f'Duplicate image paths in {split}'
    assert all(Path(p).exists() for p in selected.image_path), 'Missing images'
    assert all(Path(p).exists() for p in selected.label_path), 'Missing labels'
assert set(kept[kept.proposed_split=='train'].image_path).isdisjoint(set(kept[kept.proposed_split=='val'].image_path))
print('\n===== PROPOSED SPLIT SUMMARY =====')
summary=kept.groupby('proposed_split').agg(images=('filename','size'), net_instances=('nets','sum'), rope_instances=('ropes','sum'))
display(summary)
print('\n===== SOURCE CAMERA SUMMARY =====')
source_summary=kept.groupby(['proposed_split','source_group']).agg(images=('filename','size'), net_instances=('nets','sum'), rope_instances=('ropes','sum')).reset_index()
display(source_summary)
print('\nExcluded images:', int((inventory.proposed_split=='excluded').sum()))
print('This is a cross-camera holdout WITHIN Bistrina, not cross-location validation.')

Original SeaClear images indexed: 8610
===== VALIDATION SOURCE GROUPS (ALL HELD OUT) =====
 - Bistrina/Bluerobotics HD
 - Bistrina/SIP-E323CV
 - Jakljan/Bluerobotics HD
 - Marseille/SIP-E323CV

===== SOURCE CLASSIFICATION =====


provenance,CleanSea_cutout_excluded,Extra_background_unverified_excluded,SeaClear_verified
old_split,,,
train,193,300,2133
val,0,0,178



===== UNKNOWN IMAGES =====


,old_split,filename



===== SOURCE-CAMERA OVERLAP BETWEEN NEW SPLITS =====
None — all SeaClear camera/source folders are disjoint

===== PROPOSED SPLIT SUMMARY =====


,images,net_instances,rope_instances
proposed_split,,,
train,1461,589,1405
val,850,371,642



===== SOURCE CAMERA SUMMARY =====


,proposed_split,source_group,images,net_instances,rope_instances
0,train,Bistrina/Paralenz Vaquita Gen 2,1324,589,1348
1,train,Jakljan/Paralenz Vaquita,19,0,0
2,train,Lokrum/Bluerobotics HD,34,0,0
3,train,Lokrum/Paralenz Vaquita Gen 2,30,0,41
4,train,Lokrum/SIP-E323CV,21,0,14
5,train,Slano/Bluerobotics HD,33,0,2
6,val,Bistrina/Bluerobotics HD,672,364,571
7,val,Bistrina/SIP-E323CV,88,7,62
8,val,Jakljan/Bluerobotics HD,18,0,9
9,val,Marseille/SIP-E323CV,72,0,0



Excluded images: 493
This is a cross-camera holdout WITHIN Bistrina, not cross-location validation.


## 3. Export manifests

Save image lists, YAML, and source-audit tables without editing source data.


In [ ]:
# Save read-only image manifests, dataset configuration, and audit tables.
# Save safe image-list manifests. No images/labels are copied or changed.
train_list = OUT / 'train_images.txt'
val_list = OUT / 'val_images.txt'
for split, manifest in [('train', train_list), ('val', val_list)]:
    paths = sorted(kept.loc[kept.proposed_split == split, 'image_path'])
    manifest.write_text('\n'.join(paths) + '\n', encoding='utf-8')

config = {
    'path': str(OUT),
    'train': str(train_list),
    'val': str(val_list),
    'nc': 2,
    'names': {0: 'Net', 1: 'Rope'}
}
with (OUT / 'data.yaml').open('w', encoding='utf-8') as f:
    yaml.safe_dump(config, f, sort_keys=False)

inventory.to_csv(OUT / 'all_images_provenance_and_assignments.csv', index=False)
source_summary.to_csv(OUT / 'source_camera_summary.csv', index=False)
summary.to_csv(OUT / 'split_summary.csv')
print('Saved:')
for f in [train_list, val_list, OUT/'data.yaml', OUT/'all_images_provenance_and_assignments.csv', OUT/'source_camera_summary.csv', OUT/'split_summary.csv']:
    print(' -', f)
print('\nNo training performed. No original files changed.')

Saved:
 - /content/drive/MyDrive/MarineGuard_Validation_Improvement/MarineGuard_v2_verified_seaclear_manifest/train_images.txt
 - /content/drive/MyDrive/MarineGuard_Validation_Improvement/MarineGuard_v2_verified_seaclear_manifest/val_images.txt
 - /content/drive/MyDrive/MarineGuard_Validation_Improvement/MarineGuard_v2_verified_seaclear_manifest/data.yaml
 - /content/drive/MyDrive/MarineGuard_Validation_Improvement/MarineGuard_v2_verified_seaclear_manifest/all_images_provenance_and_assignments.csv
 - /content/drive/MyDrive/MarineGuard_Validation_Improvement/MarineGuard_v2_verified_seaclear_manifest/source_camera_summary.csv
 - /content/drive/MyDrive/MarineGuard_Validation_Improvement/MarineGuard_v2_verified_seaclear_manifest/split_summary.csv

No training performed. No original files changed.


## 4. Split outcome

Training: 1,461 images (589 Net, 1,405 Rope). Validation: 850 images (371 Net, 642 Rope). Excluded: 193 CleanSea samples and 300 additional background images.
